# OPTIMA — Kaggle: GitHub Repository → base.json → Enrichment + RAG Evaluation

This notebook has two halves:

```
PART A (new)                              PART B (unchanged, already validated on Kaggle)
================================          ==========================================
Target C/C++ GitHub repository            base.json
  -> Kaggle clones optima_python            -> LLM enrichment (1 -> 3 -> 96 functions)
  -> Kaggle clones the target repo            -> embeddings + FAISS indices
  -> Optima analyzer runs                      -> retrieval smoke test
  -> base.json is generated + validated        -> Recall@K / MRR evaluation
                                                 -> experiment outputs
```

**Part A is new.** It makes the analyzer stage configurable so any public C/C++
GitHub repository can be turned into a `base.json`, instead of relying on a
`base.json` that was already committed to the repository.

**Part B is the existing, already-working pipeline** (enrichment through evaluation).
It is reused as-is — same functions, same call signatures, same checkpoint/resume
behavior — the only change is that `BASE_JSON` now points at a freshly generated file
instead of a pre-committed one. Nothing in Part B's logic was rewritten.

Every cell does one thing, prints a clear status, and is safe to rerun. Hard gates stop
the notebook before any expensive stage (analyzer on a huge repo, 96-function
enrichment, embedding) if a prior check failed.


## Cell 1 — Configuration


In [ ]:
from pathlib import Path

# ---- Optima Python (this project) ----
OPTIMA_PYTHON_URL = "https://github.com/I1gorr/optima_python.git"
OPTIMA_PYTHON_REF = "main"  # branch, tag, or commit SHA

# ---- Target C/C++ repository to analyze (edit this to point at any project) ----
# Default is the repo Optima's own test suite is vendored from — a real, previously
# verified example (164 functions / 9 classes extracted), not a placeholder.
TARGET_REPOSITORY_URL = "https://github.com/FedericoSaitta/Chess-Engine-in-cpp.git"
TARGET_REPOSITORY_REF = "main"

# ---- Filesystem roots ----
WORK_ROOT = Path("/kaggle/working")
PROJECT_ROOT = WORK_ROOT / "optima-python"     # optima_python checkout
TARGET_ROOT = WORK_ROOT / "target-project"     # target C/C++ repo checkout
OUTPUT_ROOT = WORK_ROOT / "optima_outputs"
BASE_JSON = OUTPUT_ROOT / "base.json"          # generated by the analyzer below

# ---- Model (edit this cell only to test a different model) ----
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = False      # default OFF — do not enable unless explicitly needed
USE_FP16 = True        # default ON for the Tesla T4
MAX_NEW_TOKENS = 768
RETRIES = 2
CONTEXT_LENGTH = 8192  # input+output token budget used to build/truncate prompts

# ---- Embedding / retrieval ----
EMBEDDING_MODEL = "bge-small"       # alias from optima.rag.embedding_simple.EMBEDDING_REGISTRY
REPRESENTATION_MODE = "hybrid"      # raw | enriched | semantic | compiler | hybrid
RETRIEVAL_K = 10
NUM_BENCHMARK_QUERIES = 20
FORCE_REBUILD_INDEX = False
FORCE_REANALYZE = False  # set True to regenerate base.json even if it already exists

# ---- Debug ----
DEBUG = True  # DEBUG cells always show raw model output; this only affects verbosity

print("Configuration loaded.")
print(f"  OPTIMA_PYTHON_URL     = {OPTIMA_PYTHON_URL}")
print(f"  OPTIMA_PYTHON_REF     = {OPTIMA_PYTHON_REF}")
print(f"  TARGET_REPOSITORY_URL = {TARGET_REPOSITORY_URL}")
print(f"  TARGET_REPOSITORY_REF = {TARGET_REPOSITORY_REF}")
print(f"  PROJECT_ROOT          = {PROJECT_ROOT}")
print(f"  TARGET_ROOT           = {TARGET_ROOT}")
print(f"  OUTPUT_ROOT           = {OUTPUT_ROOT}")
print(f"  BASE_JSON             = {BASE_JSON}")
print(f"  MODEL_ID              = {MODEL_ID}")
print(f"  USE_4BIT              = {USE_4BIT}")
print(f"  USE_FP16              = {USE_FP16}")
print(f"  MAX_NEW_TOKENS        = {MAX_NEW_TOKENS}")
print(f"  RETRIES               = {RETRIES}")
print(f"  CONTEXT_LENGTH        = {CONTEXT_LENGTH}")
print(f"  EMBEDDING_MODEL       = {EMBEDDING_MODEL}")
print(f"  REPRESENTATION_MODE   = {REPRESENTATION_MODE}")
print(f"  RETRIEVAL_K           = {RETRIEVAL_K}")
print(f"  FORCE_REANALYZE       = {FORCE_REANALYZE}")


In [ ]:
# Derived paths + output directory layout. Safe to rerun.
MODEL_NAME = MODEL_ID.split("/")[-1].replace("_", "-").lower()

INPUT_DIR = OUTPUT_ROOT / "inputs"
ENRICHMENT_DIR = OUTPUT_ROOT / "enrichment"
INDICES_DIR = OUTPUT_ROOT / "indices"
EVALUATION_DIR = OUTPUT_ROOT / "evaluation"
EXPERIMENTS_DIR = OUTPUT_ROOT / "experiments"
LOGS_DIR = OUTPUT_ROOT / "logs"
WORKSPACE_DIR = INPUT_DIR / "workspace"

for directory in (INPUT_DIR, ENRICHMENT_DIR, INDICES_DIR, EVALUATION_DIR,
                   EXPERIMENTS_DIR, LOGS_DIR, WORKSPACE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Checkpoint files
DIAGNOSTIC_JSON = ENRICHMENT_DIR / "diagnostic_enriched.json"  # 1-function/3-function tests only
ENRICHED_JSON = ENRICHMENT_DIR / "enriched.json"                # full incremental run

print(f"MODEL_NAME  = {MODEL_NAME}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")
print("Created:")
for directory in (INPUT_DIR, ENRICHMENT_DIR, INDICES_DIR, EVALUATION_DIR,
                   EXPERIMENTS_DIR, LOGS_DIR, WORKSPACE_DIR):
    print(f"  {directory}")


## Cell 2 — Kaggle environment


In [ ]:
import platform
import sys

import torch

print(f"Python version:      {platform.python_version()}")
print(f"PyTorch version:      {torch.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA version:         {torch.version.cuda}")
    print(f"GPU:                  {gpu_name}")
    print(f"GPU VRAM:             {vram_gb:.2f} GiB")
else:
    print("GPU:                  none")
    print("GPU VRAM:             none")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError:
    print("Transformers version: NOT INSTALLED")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is available in this Kaggle session. Enable a GPU accelerator "
        "(Settings -> Accelerator -> GPU T4 x2 or P100) before continuing."
    )
print("\nGPU check: PASSED")


## Cell 3 — Clone Optima Python


In [ ]:
import subprocess


def _run(cmd, **kwargs):
    print(f"$ {' '.join(cmd)}")
    return subprocess.run(cmd, check=True, capture_output=True, text=True, **kwargs)


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").is_dir()


def _clone(url: str, ref: str, dest: Path):
    """Clone url@ref into dest. Reused for both optima_python and the target repo."""
    if _is_git_repo(dest):
        print(f"Repository already present at {dest}; skipping clone.")
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} exists but is not a git repository. Remove or rename it "
            "manually, then rerun this cell."
        )
    try:
        _run(["git", "clone", "--depth", "1", "--branch", ref, url, str(dest)])
    except subprocess.CalledProcessError as exc:
        # ref may be a commit SHA rather than a branch/tag name.
        print("Branch/tag clone failed; retrying as a full clone + checkout "
              f"(reason: {exc.stderr.strip()[-300:]})")
        try:
            _run(["git", "clone", url, str(dest)])
            _run(["git", "-C", str(dest), "checkout", ref])
        except subprocess.CalledProcessError as exc2:
            raise RuntimeError(
                f"Could not clone {url}. This is usually a transient GitHub "
                f"DNS/network failure on Kaggle. Underlying error:\n{exc2.stderr}"
            ) from exc2


_clone(OPTIMA_PYTHON_URL, OPTIMA_PYTHON_REF, PROJECT_ROOT)
commit = _run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"]).stdout.strip()
print(f"\u2713 repository exists: {PROJECT_ROOT}")
print(f"\u2713 repository path:   {PROJECT_ROOT}")
print(f"\u2713 commit:            {commit}")


## Cell 4 — Inspect Optima Python repository


In [ ]:
def _list(path: Path, pattern: str = "*.py"):
    return sorted(p.relative_to(PROJECT_ROOT) for p in path.rglob(pattern) if p.is_file())


print(f"Top-level entries in {PROJECT_ROOT}:")
for entry in sorted(PROJECT_ROOT.iterdir()):
    marker = "/" if entry.is_dir() else ""
    print(f"  {entry.name}{marker}")

targets = {
    "optima package (core: analyzer, enricher, CLI)": PROJECT_ROOT / "optima",
    "optima.rag (embedding/retrieval/evaluation)": PROJECT_ROOT / "optima" / "rag",
    "colab helpers (Transformers-based enrichment adapter)": PROJECT_ROOT / "colab",
    "evaluation package (metrics/analysis)": PROJECT_ROOT / "evaluation",
}
print()
for label, path in targets.items():
    status = "FOUND" if path.is_dir() else "MISSING"
    print(f"[{status}] {label}: {path}")
    if path.is_dir():
        for f in _list(path):
            print(f"    {f}")

analyzer_py = PROJECT_ROOT / "optima" / "analyzer.py"
enricher_py = PROJECT_ROOT / "optima" / "enricher.py"
pipeline_py = PROJECT_ROOT / "colab" / "colab_pipeline.py"
print()
print(f"Analyzer (base.json generator):    {analyzer_py} ({'FOUND' if analyzer_py.exists() else 'MISSING'})")
print(f"Enrichment implementation:         {enricher_py} ({'FOUND' if enricher_py.exists() else 'MISSING'})")
print(f"Kaggle/Colab Transformers adapter: {pipeline_py} ({'FOUND' if pipeline_py.exists() else 'MISSING'})")
for required in (analyzer_py, pipeline_py):
    if not required.exists():
        raise RuntimeError(f"{required} is missing from the cloned repository.")


## Cell 5 — Install Optima Python


In [ ]:
import sys

pyproject = PROJECT_ROOT / "pyproject.toml"
print(f"Inspecting {pyproject}:")
print(pyproject.read_text(encoding="utf-8"))


In [ ]:
# Editable install. Pulls the base dependencies declared in pyproject.toml (langchain,
# faiss-cpu, numpy, and the 'clang' libclang bindings the analyzer needs). Does NOT
# install the optional 'kaggle' extra (which bundles bitsandbytes) since USE_4BIT=False.
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("pip install -e of optima_python failed; see output above.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import optima

print("\u2713 optima imported")
print(f"Optima location: {optima.__file__}")


## Cell 6 — Verify imports


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))  # ensure colab_pipeline is importable as a top-level package

from optima.analyzer import analyze_project
from colab.colab_pipeline import (
    build_enrichment_messages,
    build_index,
    build_raw_index,
    enrich_nodes,
    evaluation_summary,
    flatten_functions,
    inspect_json,
    load_json,
    load_model,
    prepare_workspace,
    retrieve,
    run_evaluation,
    save_experiment_summary,
    save_json,
    validate_json,
    zip_outputs,
    _generate,              # exposes the raw model call for diagnostics
    _parse_json_object,     # exact JSON extraction logic used by enrich_nodes
    _valid_enrichment,      # exact schema validation used by enrich_nodes
)
from optima.rag.embedding_simple import EMBEDDING_REGISTRY, get_embedding_registry
from optima.rag.evaluation import load_corpus_indices

apis = {
    "Analyzer (base.json generation)": analyze_project,
    "JSON loading": load_json,
    "JSON validation": validate_json,
    "JSON inspection": inspect_json,
    "Enrichment prompt builder": build_enrichment_messages,
    "Enrichment (single/incremental)": enrich_nodes,
    "Model loader": load_model,
    "Embedding index (enriched corpus)": build_index,
    "Embedding index (raw corpus)": build_raw_index,
    "Retrieval": retrieve,
    "Corpus index loader": load_corpus_indices,
    "Evaluation (Recall@K / MRR)": run_evaluation,
    "Experiment summary writer": save_experiment_summary,
    "Output packaging": zip_outputs,
}
for label, func in apis.items():
    print(f"\u2713 {label}: {func.__module__}.{func.__name__}")


## Cell 7 — Check Clang/LLVM/libclang dependencies


In [ ]:
import shutil

# The analyzer needs three things: the 'clang' Python bindings (installed by Cell 6
# above, declared in pyproject.toml), the libclang.so the bindings load at runtime,
# and the clang/clang++ *binaries* the analyzer shells out to for LLVM IR emission.
import clang.cindex
print(f"clang.cindex (python bindings): {clang.cindex.__file__}")

clang_bin = shutil.which("clang")
clangpp_bin = shutil.which("clang++")
print(f"clang binary:   {clang_bin or 'NOT FOUND'}")
print(f"clang++ binary: {clangpp_bin or 'NOT FOUND'}")

if not (clang_bin and clangpp_bin):
    print("\nclang/clang++ binaries were not found. Installing only the 'clang' apt "
          "package (the analyzer shells out to it to emit LLVM IR) ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "clang"], check=True)
    clang_bin = shutil.which("clang")
    clangpp_bin = shutil.which("clang++")
    print(f"clang binary:   {clang_bin or 'STILL NOT FOUND'}")
    print(f"clang++ binary: {clangpp_bin or 'STILL NOT FOUND'}")
    if not (clang_bin and clangpp_bin):
        raise RuntimeError("clang/clang++ are still unavailable after apt-get install.")

from optima.analyzer import _configure_clang
_configure_clang()
index = clang.cindex.Index.create()
print("\n\u2713 libclang loaded and clang.cindex.Index.create() succeeded")


## Cell 8 — Clone target repository


In [ ]:
_clone(TARGET_REPOSITORY_URL, TARGET_REPOSITORY_REF, TARGET_ROOT)
target_commit = _run(["git", "-C", str(TARGET_ROOT), "rev-parse", "HEAD"]).stdout.strip()
print(f"\u2713 target repository exists: {TARGET_ROOT}")
print(f"\u2713 source URL:               {TARGET_REPOSITORY_URL}")
print(f"\u2713 commit:                   {target_commit}")


## Cell 9 — Verify target repository


In [ ]:
_cpp_extensions = {".c", ".cpp", ".cc", ".cxx", ".h", ".hpp", ".hh", ".hxx"}
_all_files = [p for p in TARGET_ROOT.rglob("*") if p.is_file() and ".git" not in p.parts]
_cpp_files = [p for p in _all_files if p.suffix.lower() in _cpp_extensions]

print(f"Path:                  {TARGET_ROOT}")
print(f"Total files:           {len(_all_files)}")
print(f"C/C++ source files:    {len(_cpp_files)}")

if not _cpp_files:
    raise RuntimeError(
        f"No C/C++ source files ({sorted(_cpp_extensions)}) were found under {TARGET_ROOT}. "
        "Check TARGET_REPOSITORY_URL/TARGET_REPOSITORY_REF in Cell 2."
    )
print("\nSample source files:")
for f in _cpp_files[:10]:
    print(f"  {f.relative_to(TARGET_ROOT)}")


## Cell 10 — Run analyzer


In [ ]:
import time

# The actual, existing Optima analyzer entry point — the same function the
# `optima analyze <project> -o <output_dir>` CLI command calls (optima/cli.py:
# analyze_command -> analyze_project). No alternate/fake analyzer is invented here.
#
# Resumable: if BASE_JSON already exists and FORCE_REANALYZE is False, re-analysis
# is skipped and the existing file is reused as-is.
print("Equivalent CLI command:")
print(f"  optima analyze {TARGET_ROOT} -o {OUTPUT_ROOT}")

if BASE_JSON.exists() and not FORCE_REANALYZE:
    print(f"\nbase.json already exists at {BASE_JSON}; skipping analysis "
          "(set FORCE_REANALYZE=True in Cell 2 to force a rerun).")
else:
    print(f"\nAnalyzing {TARGET_ROOT} ...")
    _analyze_start = time.perf_counter()
    generated_path = analyze_project(TARGET_ROOT, OUTPUT_ROOT)
    _analyze_seconds = time.perf_counter() - _analyze_start
    print(f"\nAnalyzer finished in {_analyze_seconds:.1f}s")
    print(f"Generated: {generated_path}")
    if generated_path != BASE_JSON:
        raise RuntimeError(
            f"Analyzer wrote base.json to {generated_path}, not the expected {BASE_JSON}."
        )


## Cell 11 — Locate generated base.json


In [ ]:
import json

print(f"Expected path: {BASE_JSON}")
if not BASE_JSON.exists():
    raise RuntimeError(f"base.json was not found at {BASE_JSON}. Check Cell 11 (Run analyzer).")
print("\u2713 exists")

if not BASE_JSON.is_file() or not (BASE_JSON.stat().st_size > 0):
    raise RuntimeError(f"{BASE_JSON} exists but is empty or not a regular file.")
print("\u2713 readable")

with BASE_JSON.open(encoding="utf-8") as handle:
    json.load(handle)
print("\u2713 valid JSON")
print(f"Path: {BASE_JSON}")


## Cell 12 — Base JSON inspection


In [ ]:
base_info = inspect_json(BASE_JSON)

num_files = base_info["files"]
num_functions = base_info["nodes"]
print(f"\nFiles:      {num_files}")
print(f"Functions:  {num_functions}")
print(f"Top-level keys: {base_info['top_level_fields']}")
print(f"Function fields ({len(base_info['function_fields'])}): {base_info['function_fields']}")

base_data = load_json(BASE_JSON)
functions = flatten_functions(base_data)
with_source = sum(1 for f in functions if f.get("source_code"))
print(f"Functions with non-empty source_code: {with_source}/{len(functions)}")
if with_source < len(functions):
    _empty_source_ids = [f.get("id") for f in functions if not f.get("source_code")][:5]
    print(f"WARNING: functions with empty source_code (sample): {_empty_source_ids}")


## Cell 13 — Base JSON validation


In [ ]:
validation = validate_json(BASE_JSON)

print(f"\nValidation status: {'OK' if validation['valid'] else 'FAILED'}")
print(f"Node count:          {validation['nodes']}")
print(f"Malformed files:     {len(validation['malformed_files'])}")
print(f"Malformed nodes:     {len(validation['malformed_nodes'])}")
print(f"Malformed enrichment:{len(validation['malformed_enrichment'])}")
print(f"Status:              {validation['status']}")

if validation["status"] != "BASE JSON":
    print("NOTE: base.json already contains enrichment fields; enrichment stages below "
          "will treat existing valid enrichment as already completed.")

if not validation["valid"]:
    raise RuntimeError(f"base.json failed structural validation: {validation}")


## Cell 14 — Extraction statistics


In [ ]:
# Analyzer-specific statistics (classes, LLVM matching, CFG, call graph) that go
# beyond generic JSON-schema validation above.
classes = [c for file in base_data.get("files", []) for c in file.get("classes", [])]
llvm_matched = sum(1 for f in functions if f.get("llvm", {}).get("matched"))
llvm_unmatched = len(functions) - llvm_matched
cfg_functions = sum(1 for f in functions if f.get("cfg", {}).get("nodes"))
cfg_edges = sum(len(f.get("cfg", {}).get("edges", [])) for f in functions)
methods = sum(1 for f in functions if f.get("class_info", {}).get("is_method"))
templates = sum(1 for f in functions if f.get("class_info", {}).get("is_template"))
call_edges = base_data.get("call_graph", {}).get("edges", [])
resolved_calls = sum(1 for f in functions for c in f.get("calls", []) if c.get("resolved"))

print(f"Files:                 {num_files}")
print(f"Functions:             {num_functions}")
print(f"  of which methods:    {methods}")
print(f"  of which templates:  {templates}")
print(f"Classes/structs:       {len(classes)}")
print(f"LLVM matched:          {llvm_matched}")
print(f"LLVM unmatched:        {llvm_unmatched}")
print(f"Functions with CFG:    {cfg_functions}")
print(f"CFG edges:             {cfg_edges}")
print(f"Call graph edges:      {len(call_edges)}")
print(f"Resolved calls:        {resolved_calls}")

if classes:
    print("\nClasses:")
    for c in classes[:15]:
        print(f"  {c['qualified_name']} ({c['kind']}) — {len(c['method_ids'])} methods, "
              f"bases={c['base_classes'] or 'none'}")


## Cell 15 — Inspect sample functions


In [ ]:
# Prints a few extracted functions in full so extraction quality can be verified by eye
# before spending any GPU time on enrichment.
_sample = functions[:5]
for i, f in enumerate(_sample, 1):
    print(f"--- sample {i}/{len(_sample)} ---")
    print(f"id:             {f.get('id')}")
    print(f"name:           {f.get('name')}")
    print(f"qualified_name: {f.get('qualified_name')}")
    print(f"source_location:{f.get('source_location')}")
    print(f"class_info:     {f.get('class_info')}")
    print(f"source_code:\n{f.get('source_code')}")
    print()


## Cell 16 — Existing enrichment pipeline: model dependency check


In [ ]:
import importlib.metadata as _metadata


def _version(pkg):
    try:
        return _metadata.version(pkg)
    except _metadata.PackageNotFoundError:
        return None


versions = {pkg: _version(pkg) for pkg in ("torch", "transformers", "accelerate", "safetensors")}
for pkg, version in versions.items():
    print(f"{pkg:<14} {version or 'NOT INSTALLED'}")

missing = [pkg for pkg, version in versions.items() if version is None]
if missing:
    print(f"\nInstalling missing required packages only: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    for pkg in missing:
        print(f"{pkg:<14} {_version(pkg)}")
else:
    print("\nAll required model-loading dependencies are already present.")

if USE_4BIT:
    print("USE_4BIT is True: bitsandbytes will be required by colab_pipeline.load_model.")
else:
    print("USE_4BIT is False: bitsandbytes is not required and was not installed.")


## Cell 17 — Load model


In [ ]:
import time

print(f"Loading {MODEL_ID} (load_in_4bit={USE_4BIT}) ...")
_load_start = time.perf_counter()
tokenizer, model = load_model(MODEL_ID, load_in_4bit=USE_4BIT)
load_seconds = time.perf_counter() - _load_start

model_dtype = next(model.parameters()).dtype
model_device = next(model.parameters()).device

print(f"\nModel:      {MODEL_ID}")
print(f"Dtype:      {model_dtype}")
print(f"Device:     {model_device}")
print(f"Load time:  {load_seconds:.1f}s")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"GPU memory allocated: {allocated:.2f} GiB")
    print(f"GPU memory reserved:  {reserved:.2f} GiB")

if USE_FP16 and model_dtype not in (torch.float16, torch.bfloat16):
    print(f"WARNING: expected a 16-bit dtype but got {model_dtype}.")


## Cell 18 — Simple model generation


In [ ]:
_prompt = "In one short sentence, what is a binary search tree?"
_messages = [{"role": "user", "content": _prompt}]
_chat_input = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)
_encoded = tokenizer(_chat_input, return_tensors="pt").to(model_device)

print(f"Input:       {_prompt!r}")
print(f"Token count: {_encoded['input_ids'].shape[1]}")

_gen_start = time.perf_counter()
with torch.inference_mode():
    _output = model.generate(
        **_encoded, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
_gen_seconds = time.perf_counter() - _gen_start
_response = tokenizer.decode(_output[0][_encoded['input_ids'].shape[1]:], skip_special_tokens=True)

print(f"Raw response:      {_response!r}")
print(f"Generation time:    {_gen_seconds:.2f}s")

if not _response.strip():
    raise RuntimeError("The model produced an empty response to a trivial prompt. "
                        "Something is wrong with model loading before Optima is even involved.")
print("\nModel smoke test: PASSED")


## Cell 19 — Select one real function


In [ ]:
SELECTED_FUNCTION = functions[0]

print(f"ID:             {SELECTED_FUNCTION.get('id')}")
print(f"Name:           {SELECTED_FUNCTION.get('name')}")
print(f"Qualified name: {SELECTED_FUNCTION.get('qualified_name')}")
_source = SELECTED_FUNCTION.get("source_code", "")
print(f"Source length:  {len(_source)} characters")
print(f"Source code:\n{_source}")

if not _source.strip():
    print("\nWARNING: source_code is EMPTY for this function.")
    print(f"Available fields on this function: {sorted(SELECTED_FUNCTION.keys())}")
    print(f"'source' field (location metadata): {SELECTED_FUNCTION.get('source')}")
    print("Inspect the fields above to find where the implementation actually lives "
          "before trusting any enrichment result for this function.")
else:
    print("\n\u2713 source_code is non-empty")


## Cell 20 — Build enrichment prompt


In [ ]:
ENRICHMENT_MESSAGES = build_enrichment_messages(SELECTED_FUNCTION)

print("System message:")
print(ENRICHMENT_MESSAGES[0]["content"])
print("\n" + "=" * 60 + "\n")
print("User message:")
print(ENRICHMENT_MESSAGES[1]["content"])

print("\n" + "=" * 60)
print("Expected JSON schema fields: purpose, behavior, summary, inputs, outputs, "
      "side_effects, dependencies, concepts, keywords, algorithm, complexity{time,space}")


## Cell 21 — Direct enrichment generation


In [ ]:
_input_ids_preview = tokenizer.apply_chat_template(
    ENRICHMENT_MESSAGES, tokenize=True, add_generation_prompt=True
)
print(f"Input tokens:        {len(_input_ids_preview)}")
print(f"Generation settings: max_new_tokens={MAX_NEW_TOKENS}, do_sample=False, "
      f"context_length={CONTEXT_LENGTH}")

_gen_start = time.perf_counter()
RAW_RESPONSE, RAW_RESPONSE_TOKENS = _generate(
    model, tokenizer, ENRICHMENT_MESSAGES, MAX_NEW_TOKENS, CONTEXT_LENGTH
)
_gen_seconds = time.perf_counter() - _gen_start

print(f"\nOutput length (chars): {len(RAW_RESPONSE)}")
print(f"Output tokens:          {RAW_RESPONSE_TOKENS}")
print(f"Generation latency:     {_gen_seconds:.2f}s")
print("\n" + "=" * 60)
print("RAW MODEL RESPONSE (unmodified):")
print("=" * 60)
print(RAW_RESPONSE)
print("=" * 60)


## Cell 22 — JSON parsing test


In [ ]:
print("Raw response:")
print(RAW_RESPONSE)

PARSED_JSON, PARSE_SOURCE = _parse_json_object(RAW_RESPONSE)
print(f"\nParse source: {PARSE_SOURCE}")
print(f"Extracted JSON: {json.dumps(PARSED_JSON, indent=2) if PARSED_JSON else None}")

if PARSED_JSON is None:
    print(f"\nJSON PARSING FAILED. Reason: {PARSE_SOURCE}")
    raise RuntimeError(
        f"Raw model response could not be parsed as JSON (reason={PARSE_SOURCE}). "
        "Inspect the raw response printed above and in the previous cell before continuing."
    )

SCHEMA_VALID, SCHEMA_REASON = _valid_enrichment(PARSED_JSON)
print(f"\nSchema valid: {SCHEMA_VALID}")
print(f"Schema reason: {SCHEMA_REASON}")

if not SCHEMA_VALID:
    raise RuntimeError(
        f"Parsed JSON failed Optima's schema validation (reason={SCHEMA_REASON}). "
        "Inspect PARSED_JSON above before continuing."
    )
print("\n\u2713 JSON parsing test PASSED")


## Cell 23 — One-function Optima enrichment


In [ ]:
_enrich_start = time.perf_counter()
diagnostic_metrics = enrich_nodes(
    BASE_JSON, DIAGNOSTIC_JSON, MODEL_ID, model, tokenizer,
    max_new_tokens=MAX_NEW_TOKENS, retries=RETRIES, limit=1, context_length=CONTEXT_LENGTH,
)
_enrich_seconds = time.perf_counter() - _enrich_start

diagnostic_data = load_json(DIAGNOSTIC_JSON)
ONE_FUNCTION_RESULT = flatten_functions(diagnostic_data)[0]
ONE_FUNCTION_ENRICHMENT = ONE_FUNCTION_RESULT.get("enrichment", {})
ONE_FUNCTION_EVAL = ONE_FUNCTION_ENRICHMENT.get("evaluation", {})

print(f"Runtime:              {_enrich_seconds:.2f}s")
print(f"Function ID:          {ONE_FUNCTION_RESULT.get('id')}")
print(f"Status:                {ONE_FUNCTION_ENRICHMENT.get('status')}")
print(f"request_success:       {ONE_FUNCTION_EVAL.get('request_success')}")
print(f"json_valid:            {ONE_FUNCTION_EVAL.get('json_valid')}")
print(f"retry_count:           {ONE_FUNCTION_EVAL.get('retry_count')}")
# enrich_nodes() records the failure reason in the returned metrics dict's
# 'failed_nodes' list, not on the per-function enrichment object itself.
_diagnostic_failures = {f["id"]: f["reason"] for f in diagnostic_metrics.get("failed_nodes", [])}
print(f"failure_reason:        {_diagnostic_failures.get(ONE_FUNCTION_RESULT.get('id'), 'n/a (succeeded)')}")
print(f"\nFinal enrichment object:")
print(json.dumps(ONE_FUNCTION_ENRICHMENT, indent=2)[:3000])


## Cell 24 — One-function success gate (HARD GATE)


In [ ]:
_required_fields = ("purpose", "behavior", "summary", "inputs", "outputs", "side_effects",
                     "dependencies", "concepts", "keywords", "algorithm", "complexity")
_missing_fields = [f for f in _required_fields if f not in ONE_FUNCTION_ENRICHMENT]

_gate_checks = {
    "enrichment object exists": bool(ONE_FUNCTION_ENRICHMENT),
    "json_valid is True": ONE_FUNCTION_EVAL.get("json_valid") is True,
    "request_success is True": ONE_FUNCTION_EVAL.get("request_success") is True,
    "status is completed": ONE_FUNCTION_ENRICHMENT.get("status") == "completed",
    "required fields present": not _missing_fields,
}

for label, passed in _gate_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {label}")

ONE_FUNCTION_GATE_PASSED = all(_gate_checks.values())

if not ONE_FUNCTION_GATE_PASSED:
    print("\n" + "=" * 60)
    print("ONE-FUNCTION ENRICHMENT FAILED.")
    print("DO NOT RUN THE FULL ENRICHMENT PIPELINE.")
    print("=" * 60)
    print(f"Missing fields: {_missing_fields}")
    print(f"Full enrichment object: {json.dumps(ONE_FUNCTION_ENRICHMENT, indent=2)}")
    raise RuntimeError(
        "One-function enrichment gate failed. Inspect the prompt/raw response/JSON "
        "parsing/enrich_nodes result cells above before rerunning."
    )
print("\n\u2713 ONE-FUNCTION ENRICHMENT GATE PASSED")


## Cell 25 — Three-function test


In [ ]:
_three_start = time.perf_counter()
three_function_metrics = enrich_nodes(
    BASE_JSON, DIAGNOSTIC_JSON, MODEL_ID, model, tokenizer,
    max_new_tokens=MAX_NEW_TOKENS, retries=RETRIES, limit=3, context_length=CONTEXT_LENGTH,
)
_three_seconds = time.perf_counter() - _three_start

diagnostic_data = load_json(DIAGNOSTIC_JSON)
THREE_FUNCTION_RESULTS = flatten_functions(diagnostic_data)[:3]

_successful = 0
_failed = 0
print(f"{'ID':<60} {'status':<12} {'latency_s':<10} {'retries':<8}")
for func in THREE_FUNCTION_RESULTS:
    enrichment = func.get("enrichment", {})
    eval_info = enrichment.get("evaluation", {})
    status = enrichment.get("status", "missing")
    latency = eval_info.get("latency_seconds")
    retries = eval_info.get("retry_count")
    _successful += status == "completed"
    _failed += status != "completed"
    print(f"{func.get('id', '')[:60]:<60} {status:<12} "
          f"{latency if latency is not None else '-':<10} {retries if retries is not None else '-':<8}")

_success_rate = _successful / len(THREE_FUNCTION_RESULTS) if THREE_FUNCTION_RESULTS else 0.0
_latencies = [
    f["enrichment"]["evaluation"]["latency_seconds"] for f in THREE_FUNCTION_RESULTS
    if f.get("enrichment", {}).get("evaluation", {}).get("latency_seconds") is not None
]
_avg_latency = sum(_latencies) / len(_latencies) if _latencies else 0.0

print(f"\nSuccessful:     {_successful}/{len(THREE_FUNCTION_RESULTS)}")
print(f"Failed:         {_failed}/{len(THREE_FUNCTION_RESULTS)}")
print(f"Success rate:   {_success_rate:.0%}")
print(f"Average latency:{_avg_latency:.2f}s")
print(f"Wall time:      {_three_seconds:.2f}s")


## Cell 26 — Three-function success gate (HARD GATE)


In [ ]:
_fallback_text = "Insufficient implementation context."
_all_fallback = all(
    f.get("enrichment", {}).get("purpose") == _fallback_text
    for f in THREE_FUNCTION_RESULTS
)

if _successful == 0 or _all_fallback:
    print("=" * 60)
    print("THREE-FUNCTION TEST FAILED.")
    print("All three functions failed or returned the same fallback response.")
    print("DO NOT RUN THE FULL ENRICHMENT PIPELINE.")
    print("=" * 60)
    raise RuntimeError(
        "Three-function enrichment gate failed "
        f"(successful={_successful}/{len(THREE_FUNCTION_RESULTS)}, all_fallback={_all_fallback})."
    )

THREE_FUNCTION_GATE_PASSED = True
print(f"\u2713 THREE-FUNCTION GATE PASSED ({_successful}/{len(THREE_FUNCTION_RESULTS)} successful)")


## Cell 27 — Prepare checkpoint


In [ ]:
TOTAL_FUNCTIONS = len(functions)

if ENRICHED_JSON.exists():
    _checkpoint_data = load_json(ENRICHED_JSON)
    _checkpoint_functions = flatten_functions(_checkpoint_data)
    _completed = sum(
        1 for f in _checkpoint_functions
        if f.get("enrichment", {}).get("status") == "completed"
    )
    print(f"Existing checkpoint found: {ENRICHED_JSON}")
else:
    _completed = 0
    print(f"No existing checkpoint; starting fresh: {ENRICHED_JSON}")

_remaining = TOTAL_FUNCTIONS - _completed
print(f"\nTotal:     {TOTAL_FUNCTIONS}")
print(f"Completed: {_completed}")
print(f"Remaining: {_remaining}")


## Cell 28 — Incremental full-corpus enrichment


In [ ]:
_full_start = time.perf_counter()
enrichment_metrics = enrich_nodes(
    BASE_JSON, ENRICHED_JSON, MODEL_ID, model, tokenizer,
    max_new_tokens=MAX_NEW_TOKENS, retries=RETRIES, limit=None, context_length=CONTEXT_LENGTH,
)
_full_seconds = time.perf_counter() - _full_start

enriched_data = load_json(ENRICHED_JSON)
enriched_functions = flatten_functions(enriched_data)

print(f"\nCompleted in {_full_seconds:.1f}s. Per-function results:")
print(f"{'#':<5} {'ID':<60} {'status':<12} {'latency_s':<10} {'retries':<8}")
for idx, func in enumerate(enriched_functions, 1):
    enrichment = func.get("enrichment", {})
    eval_info = enrichment.get("evaluation", {})
    status = enrichment.get("status", "missing")
    latency = eval_info.get("latency_seconds")
    retries = eval_info.get("retry_count")
    print(f"[{idx:02d}/{TOTAL_FUNCTIONS}] {func.get('id', '')[:60]:<60} {status:<12} "
          f"{latency if latency is not None else '-':<10} {retries if retries is not None else '-':<8}")


## Cell 29 — Enrichment summary


In [ ]:
_successful = sum(1 for f in enriched_functions if f.get("enrichment", {}).get("status") == "completed")
_failed = len(enriched_functions) - _successful

print(f"Total:            {len(enriched_functions)}")
print(f"Successful:        {_successful}")
print(f"Failed:            {_failed}")
print(f"Skipped (resumed): {enrichment_metrics.get('resumed_functions', 0)}")
print(f"Success rate:      {_successful / len(enriched_functions):.1%}" if enriched_functions else "n/a")
print(f"Average latency:   {enrichment_metrics.get('average_latency_seconds', 0):.2f}s")
print(f"Total runtime:     {enrichment_metrics.get('total_runtime_seconds', 0):.1f}s")

_summary_path = LOGS_DIR / "enrichment_summary.json"
save_json(enrichment_metrics, _summary_path)
print(f"\nSummary saved: {_summary_path}")


## Cell 30 — Resume verification


In [ ]:
_verify_data = load_json(ENRICHED_JSON)
_verify_functions = flatten_functions(_verify_data)
_verify_completed = sum(
    1 for f in _verify_functions if f.get("enrichment", {}).get("status") == "completed"
)
_verify_remaining = len(_verify_functions) - _verify_completed

print(f"{len(_verify_functions)} total")
print(f"{_verify_completed} completed")
print(f"{_verify_remaining} remaining")

if _verify_completed == 0:
    raise RuntimeError("Checkpoint shows zero completed functions after the enrichment run.")
print("\n\u2713 checkpoint correctly recognizes completed functions")


## Cell 31 — Final enriched JSON validation


In [ ]:
final_validation = validate_json(ENRICHED_JSON)
_failed_enrichment_count = sum(
    1 for f in _verify_functions if f.get("enrichment", {}).get("status") != "completed"
)

print(f"JSON valid:              {final_validation['valid']}")
print(f"Node count:              {final_validation['nodes']}")
print(f"Enrichment fields found: {final_validation['enrichment_fields']}")
print(f"Malformed nodes:         {len(final_validation['malformed_nodes'])}")
print(f"Malformed enrichment:    {len(final_validation['malformed_enrichment'])}")
print(f"Failed enrichment count: {_failed_enrichment_count}")
print(f"Status:                  {final_validation['status']}")

ENRICHED_JSON_VALID = final_validation["valid"] and final_validation["status"] != "BASE JSON"
if not ENRICHED_JSON_VALID:
    raise RuntimeError(
        f"Enriched JSON failed validation: {final_validation}. "
        "Do not continue to embeddings until this is resolved."
    )
print("\n\u2713 enriched JSON is valid; continuing to embeddings")


## Cell 32 — Embedding registry


In [ ]:
print("Available embedding aliases:")
for alias, spec in get_embedding_registry(include_mock=False).items():
    print(f"  {alias:<12} -> {spec.model_name}")

if EMBEDDING_MODEL not in EMBEDDING_REGISTRY:
    raise RuntimeError(
        f"EMBEDDING_MODEL={EMBEDDING_MODEL!r} is not a registered alias. "
        f"Choose one of: {list(EMBEDDING_REGISTRY)}"
    )
print(f"\nSelected embedding model: {EMBEDDING_MODEL} -> {EMBEDDING_REGISTRY[EMBEDDING_MODEL].model_name}")

try:
    import sentence_transformers  # noqa: F401
    print("sentence-transformers: already installed")
except ImportError:
    print("sentence-transformers: installing (required by langchain_huggingface embeddings)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)


## Cell 33 — Build embedding index


In [ ]:
prepare_workspace(BASE_JSON, ENRICHED_JSON, WORKSPACE_DIR, MODEL_NAME)

raw_index_result = build_raw_index(
    BASE_JSON, EMBEDDING_MODEL, INDICES_DIR, REPRESENTATION_MODE, FORCE_REBUILD_INDEX,
)
enriched_index_result = build_index(
    ENRICHED_JSON, MODEL_NAME, EMBEDDING_MODEL, INDICES_DIR, REPRESENTATION_MODE, FORCE_REBUILD_INDEX,
)

for label, result in (("raw", raw_index_result), ("enriched", enriched_index_result)):
    print(f"\n[{label}]")
    print(f"  embedding time (s): {result.get('elapsed_time_seconds')}")
    print(f"  vectors:            {result.get('index_metrics', {}).get('num_chunks_in_index', result.get('metadata', {}).get('num_chunks'))}")
    print(f"  dimension:          {result.get('metadata', {}).get('dimension')}")
    print(f"  index size (MB):    {result.get('index_metrics', {}).get('index_size_mb')}")
    print(f"  index path:         {result.get('index_metrics', {}).get('index_path')}")


## Cell 34 — Retrieval smoke test


In [ ]:
corpus_indices = load_corpus_indices(INDICES_DIR, EMBEDDING_MODEL)
print(f"Loaded corpora: {list(corpus_indices)}")

if MODEL_NAME not in corpus_indices:
    raise RuntimeError(f"Expected corpus '{MODEL_NAME}' not found among {list(corpus_indices)}")

_smoke_queries = [
    "What does this function do?",
    "function that evaluates a chess position",
    "parse input and validate arguments",
]

store = corpus_indices[MODEL_NAME]
for query in _smoke_queries:
    print(f"\nQuery: {query!r}")
    results = store.similarity_search_with_score(query, k=5)
    for rank, (doc, score) in enumerate(results, 1):
        function_id = doc.metadata.get("function_id", "")
        print(f"  rank={rank} function_id={function_id} score={score:.4f}")

if not _smoke_queries:
    raise RuntimeError("No smoke queries were run.")
print("\n\u2713 retrieval smoke test PASSED")


## Cell 35 — Retrieval evaluation


In [ ]:
evaluation_matrix = run_evaluation(
    INDICES_DIR,
    [EMBEDDING_MODEL],
    benchmark_source=WORKSPACE_DIR,
    num_queries=NUM_BENCHMARK_QUERIES,
    k=RETRIEVAL_K,
    output_dir=EVALUATION_DIR,
)

for embedding, corpora in evaluation_matrix.items():
    for corpus, metrics in corpora.items():
        if "error" in metrics:
            print(f"[{embedding}/{corpus}] ERROR: {metrics['error']}")
            continue
        print(f"\n[{embedding}/{corpus}]")
        print(f"  Recall@1:  {metrics.get('recall_at_1')}")
        print(f"  Recall@5:  {metrics.get('recall_at_5')}")
        print(f"  Recall@10: {metrics.get('recall_at_10')}")
        print(f"  MRR:       {metrics.get('mrr')}")
        print(f"  Mean latency: {metrics.get('mean_latency')}")

EVALUATION_ROWS = evaluation_summary(evaluation_matrix)["rows"]
PRIMARY_EVALUATION_ROW = next((r for r in EVALUATION_ROWS if r["corpus"] != "raw"), EVALUATION_ROWS[0])
print(f"\nPrimary (enriched) corpus row: {PRIMARY_EVALUATION_ROW}")


## Cell 36 — Experiment summary


In [ ]:
experiment_summary_path = EXPERIMENTS_DIR / "experiment_summary.json"
save_experiment_summary(
    experiment_summary_path,
    model_id=MODEL_ID,
    embedding_model=EMBEDDING_MODEL,
    representation_mode=REPRESENTATION_MODE,
    input_json=BASE_JSON,
    nodes=TOTAL_FUNCTIONS,
    enrichment_metrics=enrichment_metrics,
    index_results=[raw_index_result, enriched_index_result],
    matrix=evaluation_matrix,
    k=RETRIEVAL_K,
    enriched_json=ENRICHED_JSON,
    output_paths={
        "enriched_json": str(ENRICHED_JSON),
        "indices": str(INDICES_DIR),
        "evaluation": str(EVALUATION_DIR),
    },
)
print(f"Experiment summary saved: {experiment_summary_path}")
print(json.dumps(load_json(experiment_summary_path), indent=2)[:2000])


## Cell 37 — Package outputs


In [ ]:
zip_path = OUTPUT_ROOT / "optima_experiment.zip"
zip_outputs(OUTPUT_ROOT, zip_path)

zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"Package created: {zip_path}")
print(f"Package size:    {zip_size_mb:.2f} MB")


## Cell 38 — Final report


In [ ]:
print("=" * 60)
print("OPTIMA KAGGLE EXPERIMENT")
print("=" * 60)

print("\nSOURCE")
print(f"Target repository: {TARGET_REPOSITORY_URL} @ {TARGET_REPOSITORY_REF}")
print(f"Commit:             {target_commit}")

print("\nENVIRONMENT")
print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GiB")
print(f"CUDA:     {torch.version.cuda}")
print(f"PyTorch:  {torch.__version__}")

print("\nDATASET")
print(f"Files:     {num_files}")
print(f"Functions: {num_functions}")
print(f"Classes:   {len(classes)}")

print("\nMODEL")
print(f"Model:         {MODEL_ID}")
print(f"Precision:     {model_dtype}")
print(f"Quantization:  {'4-bit' if USE_4BIT else 'none'}")

print("\nENRICHMENT")
print(f"Successful:      {_successful}")
print(f"Failed:          {_failed}")
print(f"Skipped:         {enrichment_metrics.get('resumed_functions', 0)}")
print(f"Success rate:    {_successful / len(enriched_functions):.1%}" if enriched_functions else "n/a")
print(f"Average latency: {enrichment_metrics.get('average_latency_seconds', 0):.2f}s")
print(f"Total runtime:   {enrichment_metrics.get('total_runtime_seconds', 0):.1f}s")

print("\nEMBEDDING")
print(f"Model:     {EMBEDDING_MODEL} ({EMBEDDING_REGISTRY[EMBEDDING_MODEL].model_name})")
print(f"Vectors:   {enriched_index_result.get('index_metrics', {}).get('num_chunks_in_index', enriched_index_result.get('metadata', {}).get('num_chunks'))}")
print(f"Dimension: {enriched_index_result.get('metadata', {}).get('dimension')}")

print("\nRETRIEVAL")
print(f"Recall@1:  {PRIMARY_EVALUATION_ROW.get('recall_at_1')}")
print(f"Recall@5:  {PRIMARY_EVALUATION_ROW.get('recall_at_5')}")
print(f"Recall@10: {PRIMARY_EVALUATION_ROW.get('recall_at_10')}")
print(f"MRR:       {PRIMARY_EVALUATION_ROW.get('mrr')}")

print("\nOUTPUT")
print(f"base.json:            {BASE_JSON}")
print(f"Enriched JSON:        {ENRICHED_JSON}")
print(f"Indices:              {INDICES_DIR}")
print(f"Evaluation:           {EVALUATION_DIR}")
print(f"Experiment summary:   {experiment_summary_path}")
print(f"Package:              {zip_path}")
print("=" * 60)
